In [ ]:
!pip install crewai -q
!pip install langchain -q
!py -m pip install -qU langchain-ollama
!pip install -U ollama
!py -m pip install markdown weasyprint -q
!py -m pip install -qU langchain
!py -m pip install --upgrade langchain langchain-core langchain-community
!py -m pip install langchain-classic

In [1]:
import pandas as pd
from scipy.stats import boxcox
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from langchain.tools import tool
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from pathlib import Path

In [2]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3.1:8b" 

llm = ChatOllama(
    model=MODEL,
    temperature=0
)

### preprocess tool

In [ ]:
def preprocess(df):
    X = df[['n_cyanos', 'co2', 'light', 'SucRatio', 'Nsample']]
    y = df['a']

    X_train, X_temp = train_test_split(X, test_size=0.3, random_state=42)
    X_val, X_test = train_test_split(X_temp, test_size=0.5, random_state=42)

    y_train = y.loc[X_train.index]
    y_val = y.loc[X_val.index]
    y_test = y.loc[X_test.index]

    X_train_bc = X_train.copy()
    X_val_bc = X_val.copy()
    X_test_bc = X_test.copy()

    boxcox_params = {}
    for col in X_train.columns:
        min_val = X_train[col].min()
        shift = 0.0
        if min_val <= 0:
            shift = -min_val + 1e-6

        X_train_shifted = X_train[col] + shift
        X_val_shifted = X_val[col] + shift
        X_test_shifted = X_test[col] + shift

        X_train_bc[col], lam = boxcox(X_train_shifted)
        X_val_bc[col] = boxcox(X_val_shifted, lam)
        X_test_bc[col] = boxcox(X_test_shifted, lam)

        boxcox_params[col] = {"lambda": lam, "shift": shift}\
    return {
        "X_train": X_train_bc.to_dict(),
        "X_val": X_val_bc.to_dict(),
        "X_test": X_test_bc.to_dict(),
        "y_train": y_train.to_dict(),
        "y_val": y_val.to_dict(),
        "y_test": y_test.to_dict()
    }


In [ ]:
@tool
def preprocess_tool(df_path: str) -> dict:
    """
    Preprocess dataset: performs split and apply Box-Cox transformation and returns the spltted dataset
    """
    df = pd.read_csv(df_path)
    return preprocess(df)

### forecast tool

In [6]:
def forecast(preprocessed_data):
    X_train = pd.DataFrame(preprocessed_data["X_train"])
    X_val = pd.DataFrame(preprocessed_data["X_val"])
    X_test = pd.DataFrame(preprocessed_data["X_test"])

    y_train = pd.Series(preprocessed_data["y_train"])
    y_val = pd.Series(preprocessed_data["y_val"])
    y_test = pd.Series(preprocessed_data["y_test"])

    xgb = XGBRegressor(
        n_estimators=800,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective='reg:squarederror',
        n_jobs=-1,
        random_state=42
    )

    xgb.fit(X_train, y_train)

    y_train_pred = xgb.predict(X_train)
    y_val_pred = xgb.predict(X_val)
    y_test_pred = xgb.predict(X_test)

    return {
        "train_r2": float(r2_score(y_train, y_train_pred)),
        "val_r2": float(r2_score(y_val, y_val_pred)),
        "test_r2": float(r2_score(y_test, y_test_pred)),
        "test_mse": float(mean_squared_error(y_test, y_test_pred))
    }


In [7]:
@tool
def forecast_tool(preprocessed_data: dict) -> dict:
    """
    Train XGBoost model and return evaluation metrics.
    """
    return forecast(preprocessed_data)

### agent orchestration

In [ ]:
preprocess_prompt = ChatPromptTemplate.from_messages([
    ("system", "You must call preprocess_tool with the provided CSV path and return splitted dataset"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

preprocess_agent = create_tool_calling_agent(
    llm,
    [preprocess_tool],
    preprocess_prompt
)

preprocess_executor = AgentExecutor(
    agent=preprocess_agent,
    tools=[preprocess_tool],
    verbose=True
)

In [9]:
forecast_prompt = ChatPromptTemplate.from_messages([
    ("system", "You must call forecast_tool using the provided preprocessed data."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

forecast_agent = create_tool_calling_agent(
    llm,
    [forecast_tool],
    forecast_prompt
)

forecast_executor = AgentExecutor(
    agent=forecast_agent,
    tools=[forecast_tool],
    verbose=True
)


In [10]:
report_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """
You are a machine learning report generator.

Given evaluation metrics, generate a professional Markdown report including:
- Model Overview
- Performance Table
- Interpretation
- Overfitting Assessment
- Conclusion
"""),
    ("human", "{input}")
])

report_chain = report_prompt | llm


In [ ]:
preprocess_result = preprocess_executor.invoke({
        "input": 'data/fitting-results.csv'
    })["output"]




> Entering new AgentExecutor chain...

Invoking: `preprocess_tool` with `{'df_path': 'data/fitting-results.csv'}`


ran2
ran3
{'X_train': {'n_cyanos': {1180: 171.16438724640668, 9254: 57.11179453108728, 6946: 152.53231388940262, 6482: 111.48321708515853, 6872: 133.91540199676666, 5422: 163.8727326753078, 3493: 144.46048248294338, 8829: 61.50696726919933, 1459: 152.15223555967134, 6440: 139.23167743850885, 8522: 111.48321708515853, 5358: 69.38227251159317, 14706: 150.50039008096542, 16348: 79.9347183623312, 6685: 26.528719414990636, 15643: 112.49790724911624, 12834: 71.15790736480442, 8019: 0.8898995465257321, 1586: 37.953181064718585, 1395: 53.167769485994654, 10796: 138.30781921802463, 7885: 133.91540199676666, 3889: 44.002395491028324, 19067: 153.41755401135418, 16508: 70.62725320691057, 19885: 29.939857764896963, 5620: 134.31729845976488, 7346: 53.167769485994654, 4851: 75.85949164443625, 6624: 69.38227251159317, 18782: 167.04957591766748, 15554: 130.27417925346927, 14660: 18.7506

In [ ]:

forecast_result = forecast_executor.invoke({
        "input": preprocess_result
    })["output"]

report = report_chain.invoke({
        "input": str(forecast_result)
    }).content

In [ ]:
with open("result/forecast_report.md", "w", encoding="utf-8") as f:
    f.write(report)

print("Report saved to forecast_report.md")


Report saved to forecast_report.md


### References
https://docs.langchain.com/oss/python/integrations/chat/ollama

https://docs.langchain.com/oss/python/langchain/tools


- feed more data about the forecasting process and feature engineering information for better report
- purpose of the task (is it to understand how agentic system works or should we try to build an efficient system for real life scenario)
- should the tool be called by the agent 


In [ ]:
# def forecast(df):
#     X = df[['n_cyanos', 'co2', 'light', 'SucRatio', 'Nsample']]
#     y_a = df['a']

#     X_train, X_temp = train_test_split(
#         X, test_size=0.3, random_state=42
#     )
#     X_val, X_test = train_test_split(
#         X_temp, test_size=0.5, random_state=42
#     )

#     y_a_train, y_a_val, y_a_test = y_a.loc[X_train.index], y_a.loc[X_val.index], y_a.loc[X_test.index]
#     X_train_bc = X_train.copy()
#     X_val_bc = X_val.copy()
#     X_test_bc = X_test.copy()

#     boxcox_params = {}  

#     for col in X_train.columns:
#         min_val = X_train[col].min()
#         shift = 0.0
#         if min_val <= 0:
#             shift = -min_val + 1e-6

#         X_train_shifted = X_train[col] + shift
#         X_val_shifted = X_val[col] + shift
#         X_test_shifted = X_test[col] + shift

#         X_train_bc[col], lam = boxcox(X_train_shifted)

#         X_val_bc[col] = boxcox(X_val_shifted, lam)
#         X_test_bc[col] = boxcox(X_test_shifted, lam)
        
#         boxcox_params[col] = {
#             "lambda": lam,
#             "shift": shift
#         }
        
#     xgb = XGBRegressor(
#         n_estimators=800,
#         learning_rate=0.05,
#         max_depth=6,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         reg_lambda=1.0,
#         objective='reg:squarederror',
#         n_jobs=-1,
#         random_state=42
#     )
#     xgb.fit(X_train_bc, y_a_train)

#     y_a_train_pred = xgb.predict(X_train_bc)
#     y_a_val_pred = xgb.predict(X_val_bc)
#     y_a_test_pred = xgb.predict(X_test_bc)
#     return {
#         "train_r2": r2_score(y_a_train, y_a_train_pred),
#         "val_r2": r2_score(y_a_val, y_a_val_pred),
#         "test_r2":  r2_score(y_a_test, y_a_test_pred),
#         "test_mse": mean_squared_error(y_a_test, y_a_test_pred)
#     }

# @tool
# def run_forecast(df_path) -> dict:
#     """
#     Runs the forecasting pipeline on the CSV file located at df_path.
#     Returns model evaluation metrics including R2 scores and MSE of train, test and validation.
#     """
#     df = pd.read_csv(df_path)
#     return forecast(df)

# prompt = ChatPromptTemplate.from_messages(
#     [
#         (
#             "system",
#             """
# You are a machine learning evaluation assistant.

# When the user provides a CSV file path:
# 1. Call the `run_forecast` tool with that path.
# 2. After receiving the metrics, generate a professional Markdown report.

# The report must include:
# - Model Overview
# - Train / Validation / Test R²
# - Test MSE
# - Interpretation of performance
# - Assessment of overfitting or generalization
# - Final conclusion

# Do not hallucinate metrics. Only use tool output.
# Return clean Markdown.
# """
#         ),
#         ("human", "{input}"),
#         ("placeholder", "{agent_scratchpad}")
#     ]
# )